# 🦆 Exploration du warehouse DuckDB

Après avoir lancé `make all`, on peut requêter les marts directement avec Python.

Ce notebook montre comment :
1. Se connecter à DuckDB en lecture seule
2. Lister les schémas et tables
3. Requêter chaque mart
4. Visualiser quelques résultats

In [ ]:
import duckdb
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style='whitegrid')
conn = duckdb.connect('../warehouse/tmdb.duckdb', read_only=True)
print('✓ Connecté à DuckDB')

## 1. Inventaire des schémas et tables

In [ ]:
conn.execute("""
    SELECT table_schema, table_name, table_type
    FROM information_schema.tables
    WHERE table_schema NOT IN ('information_schema', 'pg_catalog')
    ORDER BY table_schema, table_name
""").fetchdf()

## 2. KPIs globaux

In [ ]:
kpis = conn.execute('SELECT * FROM main_marts.mart_global_kpis').fetchdf().T
kpis.columns = ['value']
kpis

## 3. Top 10 personnes

In [ ]:
conn.execute('''
    SELECT rank, person_name, department, popularity_score, gender
    FROM main_marts.mart_top_100
    LIMIT 10
''').fetchdf()

## 4. Stats par département

In [ ]:
dept = conn.execute('SELECT * FROM main_marts.dim_department_stats').fetchdf()
dept

## 5. Heatmap diversité

In [ ]:
matrix = conn.execute('SELECT * FROM main_marts.mart_department_gender_matrix').fetchdf()
pivot = matrix.pivot(index='department', columns='gender', values='n_people').fillna(0).astype(int)

fig, ax = plt.subplots(figsize=(10, 6))
sns.heatmap(pivot, annot=True, fmt='d', cmap='YlOrRd', ax=ax)
ax.set_title('Heatmap département × genre')
plt.show()

## 6. Requêtes ad-hoc — exemple : femmes dans Directing

In [ ]:
conn.execute('''
    SELECT person_name, popularity_score, popularity_rank_in_department
    FROM main_marts.fct_people
    WHERE department = 'Directing' AND gender = 'female'
    ORDER BY popularity_score DESC
    LIMIT 10
''').fetchdf()

In [ ]:
conn.close()
print('✓ Connexion fermée')